# Flocking in continuous space: one tree versus n scans

The bundled Boids model answers neighbour queries for the whole flock with one KD-tree per step (`neighbors_within`). The `ScanBoids` variant below asks per agent instead, the O(n^2) pattern, to show where the tree starts to matter.

In [1]:
import json
import time

import pandas as pd


def bench(label, make_model, steps, agents_of=lambda m: len(m.agents)):
    """Build the model, time ``steps`` steps, print a RESULT line, return a row."""
    model = make_model()
    n = agents_of(model)
    t0 = time.perf_counter()
    model.run_for(steps)
    per_step = (time.perf_counter() - t0) / steps
    row = {
        "label": label,
        "agents": n,
        "ms_per_step": round(per_step * 1e3, 2),
        "us_per_agent_step": round(per_step / max(n, 1) * 1e6, 2),
    }
    print("RESULT " + json.dumps(row))
    return row

In [2]:
from modelly.examples.basic.boid_flockers.agents import Boid
from modelly.examples.basic.boid_flockers.model import BoidFlockers, BoidsScenario


class ScanBoid(Boid):
    """A boid that scans the whole flock for its neighbours (O(n) per agent)."""

    def step(self):
        neighbors, _ = self.get_neighbors_in_radius(radius=self.vision)
        self.model.neighbor_lists = {self: neighbors}
        super().step()


class ScanBoids(BoidFlockers):
    """Boids without the per-step tree: every agent scans every position."""

    def step(self):
        self.agents.shuffle_do("step")
        self.update_average_heading()


def tree(n, side):
    return lambda: BoidFlockers(BoidsScenario(population_size=n, width=side, height=side, vision=10.0, rng=1))


def scan(n, side):
    def make():
        model = ScanBoids(BoidsScenario(population_size=n, width=side, height=side, vision=10.0, rng=1))
        for boid in model.agents:
            boid.__class__ = ScanBoid
        return model
    return make

In [3]:
rows = []
rows.append(bench("500 boids, one tree per step", tree(500, 100), steps=10))
rows.append(bench("500 boids, one scan per agent", scan(500, 100), steps=10))
rows.append(bench("2000 boids, one tree per step", tree(2000, 200), steps=5))
rows.append(bench("2000 boids, one scan per agent", scan(2000, 200), steps=5))
rows.append(bench("5000 boids, one tree per step", tree(5000, 300), steps=3))

RESULT {"label": "500 boids, one tree per step", "agents": 500, "ms_per_step": 14.69, "us_per_agent_step": 29.37}


RESULT {"label": "500 boids, one scan per agent", "agents": 500, "ms_per_step": 20.62, "us_per_agent_step": 41.25}


RESULT {"label": "2000 boids, one tree per step", "agents": 2000, "ms_per_step": 54.58, "us_per_agent_step": 27.29}


RESULT {"label": "2000 boids, one scan per agent", "agents": 2000, "ms_per_step": 126.49, "us_per_agent_step": 63.24}


RESULT {"label": "5000 boids, one tree per step", "agents": 5000, "ms_per_step": 245.65, "us_per_agent_step": 49.13}


In [4]:
pd.DataFrame(rows).set_index('label')

,agents,ms_per_step,us_per_agent_step
label,,,
"500 boids, one tree per step",500,14.69,29.37
"500 boids, one scan per agent",500,20.62,41.25
"2000 boids, one tree per step",2000,54.58,27.29
"2000 boids, one scan per agent",2000,126.49,63.24
"5000 boids, one tree per step",5000,245.65,49.13
